# 5.4. 自定义层

当框架中没有现成层时，可通过继承 nn.Module 构建自定义层。请自己完成每个代码单元。

导入所需模块：torch、torch.nn、torch.nn.functional（F）。

In [2]:
import torch
import torch.nn.functional as F
from torch import nn

### 5.4.1. 不带参数的层

实现一个 CenteredLayer，从其输入中减去均值：
- 继承 nn.Module，__init__ 中调用 super().__init__()
- def forward(self, X): return X - X.mean()

测试：layer = CenteredLayer()；layer(torch.FloatTensor([1, 2, 3, 4, 5]))。

In [3]:

class CenteredLayer(nn.Module):
    def __init__(self) -> None:
        super().__init__()
    def forward(self,X):
        return X-X.mean()


把该层作为组件合并进更复杂的模型：
- net = nn.Sequential(nn.Linear(8, 128), CenteredLayer())
- Y = net(torch.rand(4, 8))，再验证 Y.mean() 是否接近 0

In [4]:
net = nn.Sequential(nn.Linear(8, 128), CenteredLayer())
Y = net(torch.rand(4, 8))
print(Y.mean())

tensor(6.9849e-09, grad_fn=<MeanBackward0>)


### 5.4.2. 带参数的层

实现自定义全连接层 MyLinear，以 ReLU 为激活，包含可训练的 weight 和 bias：
- class MyLinear(nn.Module):
  - __init__(self, in_units, units)：super().__init__()；self.weight = nn.Parameter(torch.randn(in_units, units))；self.bias = nn.Parameter(torch.randn(units,))
  - def forward(self, X): linear = torch.matmul(X, self.weight.data) + self.bias.data
  - return F.relu(linear)

实例化 linear = MyLinear(5, 3) 并查看 linear.weight。

In [5]:

class MyLinear(nn.Module):
    def __init__(self, in_units, units) -> None:
        super().__init__()
        self.weight=nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))
    def forward(self,X):
        linear=torch.matmul(X,self.weight.data)+self.bias.data
        return F.relu(linear)
linear=MyLinear(5,3)
print(linear.weight)

Parameter containing:
tensor([[ 0.9098,  0.5800,  0.9183],
        [-1.1180, -1.2194, -0.4461],
        [-0.1725,  1.4721,  0.2308],
        [-1.2584, -1.5089, -0.6697],
        [-1.3523, -0.0979, -1.1159]], requires_grad=True)


使用自定义层进行前向传播，并构建模型：
- linear(torch.rand(2, 5)) 直接计算
- net = nn.Sequential(MyLinear(64, 8), MyLinear(8, 1))；net(torch.rand(2, 64))

In [7]:
linear(torch.rand(2,5))
net=nn.Sequential(MyLinear(64,8),MyLinear(8,1))
net(torch.rand(2,64))


tensor([[0.],
        [0.]])

### 5.4.3. 小结

- 可以通过继承基础层类设计行为各异的自定义层
- 自定义层定义完成后，可在任意环境与网络架构中调用
- 层可以有局部参数，通过内置函数创建（nn.Parameter）

### 5.4.4. 练习

1. 设计一个接受输入并计算张量降维的层，其输出 y_k = Σ_ij W_kij * x_i * x_j。
2. 设计一个返回输入数据傅立叶系数前半部分的层。

In [21]:



class Layer1(nn.Module):
    def __init__(self,output_dim:int, input_dim:int) -> None:
        super().__init__()
        self.w=nn.Parameter(torch.randn(output_dim, input_dim, input_dim))
    def forward(self,x):
           return torch.einsum("bi,kij,bj->bk", x, self.w, x)
linear1=Layer1(3,3)
print(linear1(torch.rand((2,3))))


tensor([[-0.0022,  0.1235,  1.0021],
        [ 0.4810,  0.4705, -0.0575]], grad_fn=<ViewBackward0>)
